# Proyecto Data Mining — Corte 1
## EA Sports FC Players: EDA + DuckDB Warehouse + ML



---
## 1. Carga de Datos

In [1]:
import os, warnings, json
import numpy as np
import pandas as pd
import duckdb
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report, roc_auc_score,
)

warnings.filterwarnings('ignore')
pd.set_option('mode.copy_on_write', False)

# ── Rutas ──
BASE_DIR   = os.path.abspath('.')
DATA_DIR   = os.path.join(BASE_DIR, 'data')
MODELS_DIR = os.path.join(BASE_DIR, 'models')
DB_PATH    = os.path.join(BASE_DIR, 'warehouse.duckdb')
CSV_PATH   = os.path.join(DATA_DIR, 'male_players.csv')

os.makedirs(DATA_DIR,   exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print(f'Base dir: {BASE_DIR}')
print(f'CSV path: {CSV_PATH}')

Base dir: /Users/jaffetvicente/Downloads/Mineria de Datos/proyecto_corte_1/pipeline
CSV path: /Users/jaffetvicente/Downloads/Mineria de Datos/proyecto_corte_1/pipeline/data/male_players.csv


In [2]:
# Cargar dataset completo
df_raw = pd.read_csv(CSV_PATH)
print(f'Shape original (todas las versiones FIFA): {df_raw.shape}')
print(f'\nVersiones FIFA disponibles: {sorted(df_raw["fifa_version"].unique())}')
print(f'\nRegistros por versión:')
print(df_raw['fifa_version'].value_counts().sort_index())

Shape original (todas las versiones FIFA): (180021, 109)

Versiones FIFA disponibles: [np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(19.0), np.float64(20.0), np.float64(21.0), np.float64(22.0), np.float64(23.0), np.float64(24.0)]

Registros por versión:
fifa_version
15.0    16182
16.0    16706
17.0    17596
18.0    17954
19.0    18086
20.0    18483
21.0    18892
22.0    19239
23.0    18533
24.0    18350
Name: count, dtype: int64


In [3]:
# Filtrar SOLO la versión más reciente de FIFA para evitar duplicados
latest_version = df_raw['fifa_version'].max()
df = df_raw[df_raw['fifa_version'] == latest_version].copy().reset_index(drop=True)
print(f'Versión seleccionada: FIFA {latest_version}')
print(f'Shape filtrado: {df.shape}')
df.head()

Versión seleccionada: FIFA 24.0
Shape filtrado: (18350, 109)


,player_id,player_url,fifa_version,fifa_update,update_as_of,short_name,long_name,player_positions,overall,potential,...,ldm,cdm,rdm,rwb,lb,lcb,cb,rcb,rb,gk
0,231747,/player/231747/kylian-mbappe/240002,24.0,2.0,2023-09-22,K. Mbappé,Kylian Mbappé Lottin,"ST, LW",91,94,...,63+3,63+3,63+3,68+3,63+3,54+3,54+3,54+3,63+3,18+3
1,239085,/player/239085/erling-haaland/240002,24.0,2.0,2023-09-22,E. Haaland,Erling Braut Haaland,ST,91,94,...,63+3,63+3,63+3,62+3,60+3,62+3,62+3,62+3,60+3,19+3
2,192985,/player/192985/kevin-de-bruyne/240002,24.0,2.0,2023-09-22,K. De Bruyne,Kevin De Bruyne,"CM, CAM",91,91,...,80+3,80+3,80+3,79+3,75+3,70+3,70+3,70+3,75+3,21+3
3,158023,/player/158023/lionel-messi/240002,24.0,2.0,2023-09-22,L. Messi,Lionel Andrés Messi Cuccittini,"CF, CAM",90,90,...,63+3,63+3,63+3,64+3,59+3,49+3,49+3,49+3,59+3,19+3
4,165153,/player/165153/karim-benzema/240002,24.0,2.0,2023-09-22,K. Benzema,Karim Benzema,"CF, ST",90,90,...,64+3,64+3,64+3,64+3,60+3,55+3,55+3,55+3,60+3,18+3


In [4]:
# Exploración inicial
print('=== INFO ===')
df.info()
print('\n=== DESCRIBE (numéricas) ===')
df.describe()

=== INFO ===
<class 'pandas.DataFrame'>
RangeIndex: 18350 entries, 0 to 18349
Columns: 109 entries, player_id to gk
dtypes: float64(20), int64(43), object(1), str(45)
memory usage: 15.3+ MB

=== DESCRIBE (numéricas) ===


,player_id,fifa_version,fifa_update,overall,potential,value_eur,wage_eur,age,height_cm,weight_kg,...,mentality_composure,defending_marking_awareness,defending_standing_tackle,defending_sliding_tackle,goalkeeping_diving,goalkeeping_handling,goalkeeping_kicking,goalkeeping_positioning,goalkeeping_reflexes,goalkeeping_speed
count,18350.000000,18350.0,18350.0,18350.000000,18350.000000,1.825000e+04,18263.000000,18350.000000,18350.000000,18350.000000,...,18350.000000,18350.000000,18350.000000,18350.000000,18350.000000,18350.000000,18350.000000,18350.000000,18350.000000,2045.000000
mean,242440.726322,24.0,2.0,65.817057,71.088065,2.837585e+06,8723.388819,25.267139,181.698747,75.210354,...,57.976240,46.666975,48.532534,46.336240,16.329373,16.124578,16.035095,16.168392,16.423815,35.253790
std,26741.238580,0.0,0.0,6.817917,6.220982,7.562794e+06,18707.237605,4.757756,6.869995,6.985703,...,12.137094,20.415339,21.046124,20.569969,17.572154,16.945031,16.699136,17.093683,17.880726,10.591029
min,18115.000000,24.0,2.0,47.000000,48.000000,1.000000e+04,500.000000,16.000000,156.000000,49.000000,...,13.000000,3.000000,8.000000,6.000000,2.000000,2.000000,2.000000,2.000000,2.000000,15.000000
25%,225442.000000,24.0,2.0,62.000000,67.000000,4.750000e+05,1000.000000,21.000000,177.000000,70.000000,...,51.000000,29.000000,29.000000,26.000000,8.000000,8.000000,8.000000,8.000000,8.000000,26.000000
50%,245467.500000,24.0,2.0,66.000000,71.000000,1.000000e+06,3000.000000,25.000000,182.000000,75.000000,...,59.000000,52.000000,56.000000,53.000000,11.000000,11.000000,11.000000,11.000000,11.000000,34.000000
75%,264169.500000,24.0,2.0,70.000000,75.000000,2.000000e+06,8000.000000,29.000000,187.000000,80.000000,...,66.000000,63.000000,65.000000,63.000000,14.000000,14.000000,14.000000,14.000000,14.000000,44.000000
max,278145.000000,24.0,2.0,91.000000,94.000000,1.850000e+08,350000.000000,43.000000,206.000000,105.000000,...,96.000000,91.000000,91.000000,90.000000,90.000000,90.000000,91.000000,90.000000,93.000000,65.000000


---
## 2. EDA — Análisis Exploratorio de Datos

### 2.1 Análisis de Valores Nulos

In [5]:
# Análisis de nulos
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_report = pd.DataFrame({'nulos': null_counts, 'porcentaje': null_pct})
null_report = null_report[null_report['nulos'] > 0].sort_values('nulos', ascending=False)
print(f'Columnas con valores nulos: {len(null_report)}')
null_report

Columnas con valores nulos: 25


,nulos,porcentaje
nation_team_id,17570,95.75
nation_jersey_number,17570,95.75
nation_position,17570,95.75
club_loaned_from,17170,93.57
player_tags,17098,93.18
goalkeeping_speed,16305,88.86
player_traits,9741,53.08
physic,2045,11.14
defending,2045,11.14
dribbling,2045,11.14


### 2.2 Limpieza y Tratamiento de Nulos

**Estrategia:**
- Eliminar columnas de baja utilidad analítica (URLs, tags, traits, ratings compuestos por posición)
- Los porteros (GK) tienen `NaN` en pace, shooting, etc. → imputar con 0
- Columnas numéricas restantes → imputar con la mediana
- Columnas categóricas con NaN → imputar con 'Unknown'

In [6]:
# Columnas de ratings por posición (ls, st, rs, ... gk) — son strings tipo '90+3'
pos_rating_cols = ['ls','st','rs','lw','lf','cf','rf','rw','lam','cam','ram',
                   'lm','lcm','cm','rcm','rm','lwb','ldm','cdm','rdm','rwb',
                   'lb','lcb','cb','rcb','rb','gk']

# Columnas a eliminar por baja utilidad analítica
drop_cols = ['player_url', 'player_tags', 'player_traits', 'club_loaned_from',
             'dob', 'update_as_of', 'fifa_update'] + pos_rating_cols

existing_drops = [c for c in drop_cols if c in df.columns]
df = df.drop(columns=existing_drops)
print(f'Columnas eliminadas: {len(existing_drops)}')
print(f'Shape tras eliminación: {df.shape}')

Columnas eliminadas: 34
Shape tras eliminación: (18350, 75)


In [7]:
# Imputación: atributos de campo para porteros → 0
field_attrs = ['pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']
for col in field_attrs:
    n_null = df[col].isnull().sum()
    if n_null > 0:
        df[col] = df[col].fillna(0)
        print(f'  {col}: {n_null} NaN → imputados con 0 (porteros)')

# Imputación: categóricas con NaN → 'Unknown'
cat_cols_raw = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols_raw:
    n_missing = df[c].isna().sum()
    if n_missing > 0:
        df[c] = df[c].fillna('Unknown')
        print(f'  {c}: {n_missing} NaN → imputados con "Unknown"')

# Imputación: numéricos restantes con mediana
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
for c in num_cols:
    n_missing = df[c].isna().sum()
    if n_missing > 0:
        median = df[c].median()
        df[c] = df[c].fillna(median)
        print(f'  {c}: {n_missing} NaN → imputados con mediana={median:.2f}')

print(f'\nNulos restantes: {df.isnull().sum().sum()}')

  pace: 2045 NaN → imputados con 0 (porteros)
  shooting: 2045 NaN → imputados con 0 (porteros)
  passing: 2045 NaN → imputados con 0 (porteros)
  dribbling: 2045 NaN → imputados con 0 (porteros)
  defending: 2045 NaN → imputados con 0 (porteros)
  physic: 2045 NaN → imputados con 0 (porteros)
  club_name: 87 NaN → imputados con "Unknown"
  league_name: 87 NaN → imputados con "Unknown"
  club_position: 87 NaN → imputados con "Unknown"
  club_joined_date: 1267 NaN → imputados con "Unknown"
  nation_position: 17570 NaN → imputados con "Unknown"
  value_eur: 100 NaN → imputados con mediana=1000000.00
  wage_eur: 87 NaN → imputados con mediana=3000.00
  club_team_id: 87 NaN → imputados con mediana=1910.00
  league_id: 87 NaN → imputados con mediana=56.00
  league_level: 87 NaN → imputados con mediana=1.00
  club_jersey_number: 87 NaN → imputados con mediana=18.00
  club_contract_valid_until_year: 87 NaN → imputados con mediana=2025.00
  nation_team_id: 17570 NaN → imputados con mediana=135

### 2.3 Distribuciones de Variables Clave

In [8]:
key_vars = ['overall', 'potential', 'value_eur', 'wage_eur', 'age',
            'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']
df[key_vars].describe().round(2)

,overall,potential,value_eur,wage_eur,age,pace,shooting,passing,dribbling,defending,physic
count,18350.00,18350.00,1.835000e+04,18350.00,18350.00,18350.00,18350.00,18350.00,18350.00,18350.00,18350.00
mean,65.82,71.09,2.827571e+06,8696.25,25.27,60.75,46.72,51.08,55.84,46.29,57.67
std,6.82,6.22,7.543371e+06,18666.98,4.76,23.79,21.11,20.36,21.69,22.29,22.46
min,47.00,48.00,1.000000e+04,500.00,16.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,62.00,67.00,4.750000e+05,1000.00,21.00,58.00,35.00,47.00,54.00,31.00,54.00
50%,66.00,71.00,1.000000e+06,3000.00,25.00,68.00,52.00,56.00,62.00,54.00,64.00
75%,70.00,75.00,2.000000e+06,8000.00,29.00,75.00,62.00,63.00,68.00,63.00,71.00
max,91.00,94.00,1.850000e+08,350000.00,43.00,97.00,93.00,94.00,94.00,89.00,89.00


### 2.4 Detección y Capping de Outliers (IQR 1.5×)

In [9]:
outlier_cols = ['value_eur', 'wage_eur', 'age', 'height_cm', 'weight_kg']
outlier_report = {}

for c in outlier_cols:
    Q1, Q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    IQR = Q3 - Q1
    lo, hi = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    mask = (df[c] < lo) | (df[c] > hi)
    n_out = mask.sum()
    if n_out > 0:
        outlier_report[c] = n_out
        df[c] = df[c].clip(lower=lo, upper=hi)

print('Outliers detectados y acotados (capping):')
for col, count in outlier_report.items():
    print(f'  {col}: {count} outliers')

Outliers detectados y acotados (capping):
  value_eur: 2091 outliers
  wage_eur: 2154 outliers
  age: 5 outliers
  height_cm: 24 outliers
  weight_kg: 75 outliers


### 2.5 Análisis de Correlaciones

In [10]:
corr_cols = ['overall', 'potential', 'value_eur', 'wage_eur', 'age',
             'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
             'skill_moves', 'weak_foot', 'international_reputation']

corr_matrix = df[corr_cols].corr().round(3)
print('Correlación con overall:')
print(corr_matrix['overall'].sort_values(ascending=False))
print('\nCorrelación con value_eur:')
print(corr_matrix['value_eur'].sort_values(ascending=False))

Correlación con overall:
overall                     1.000
value_eur                   0.826
wage_eur                    0.751
potential                   0.653
international_reputation    0.469
age                         0.447
passing                     0.373
skill_moves                 0.338
shooting                    0.337
dribbling                   0.322
defending                   0.281
physic                      0.276
weak_foot                   0.212
pace                        0.140
Name: overall, dtype: float64

Correlación con value_eur:
value_eur                   1.000
overall                     0.826
wage_eur                    0.789
potential                   0.779
international_reputation    0.414
passing                     0.373
skill_moves                 0.356
dribbling                   0.352
shooting                    0.343
physic                      0.249
defending                   0.248
pace                        0.221
weak_foot                   0.205

### 2.6 Codificación de Variables Categóricas

In [11]:
# Guardar copia para el Data Warehouse ANTES de codificar
df_warehouse = df.copy()

# Columnas categóricas a codificar para ML
cat_encode_cols = ['preferred_foot', 'work_rate', 'body_type', 'real_face']

le_dict = {}
for c in cat_encode_cols:
    if c in df.columns:
        le = LabelEncoder()
        df[c] = le.fit_transform(df[c].astype(str))
        le_dict[c] = le
        print(f'  {c}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

print(f'\nColumnas codificadas: {list(le_dict.keys())}')

  preferred_foot: {'Left': np.int64(0), 'Right': np.int64(1)}
  work_rate: {'High/High': np.int64(0), 'High/Low': np.int64(1), 'High/Medium': np.int64(2), 'Low/High': np.int64(3), 'Low/Low': np.int64(4), 'Low/Medium': np.int64(5), 'Medium/High': np.int64(6), 'Medium/Low': np.int64(7), 'Medium/Medium': np.int64(8)}
  body_type: {'Lean (170-)': np.int64(0), 'Lean (170-185)': np.int64(1), 'Lean (185+)': np.int64(2), 'Normal (170-)': np.int64(3), 'Normal (170-185)': np.int64(4), 'Normal (185+)': np.int64(5), 'Stocky (170-)': np.int64(6), 'Stocky (170-185)': np.int64(7), 'Stocky (185+)': np.int64(8), 'Unique': np.int64(9)}
  real_face: {'No': np.int64(0), 'Yes': np.int64(1)}

Columnas codificadas: ['preferred_foot', 'work_rate', 'body_type', 'real_face']


### 2.7 Normalización con StandardScaler

In [12]:
exclude_from_scaling = ['player_id', 'club_team_id', 'league_id', 'nationality_id',
                        'nation_team_id', 'fifa_version']
scale_cols = [c for c in df.select_dtypes(include=[np.number]).columns
              if c not in exclude_from_scaling]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

joblib.dump(scaler, os.path.join(MODELS_DIR, 'scaler.joblib'))
print(f'Scaler guardado → {MODELS_DIR}/scaler.joblib')
print(f'Columnas escaladas: {len(scale_cols)}')

Scaler guardado → /Users/jaffetvicente/Downloads/Mineria de Datos/proyecto_corte_1/pipeline/models/scaler.joblib
Columnas escaladas: 60


In [13]:
clean_path = os.path.join(DATA_DIR, 'players_clean.csv')
df.to_csv(clean_path, index=False)
print(f'Dataset limpio guardado → {clean_path}')
print(f'Shape final: {df.shape}')
print(f'NaN totales: {df.isnull().sum().sum()}')

Dataset limpio guardado → /Users/jaffetvicente/Downloads/Mineria de Datos/proyecto_corte_1/pipeline/data/players_clean.csv
Shape final: (18350, 75)
NaN totales: 0


---
## 3. Data Warehouse — DuckDB (Esquema Estrella)

- **dim_jugador**: datos demográficos del jugador
- **dim_club**: información del club y liga
- **fact_rendimiento**: métricas de rendimiento, valor y potencial
- Vistas OLAP pre-calculadas

In [14]:
raw = df_warehouse.copy()
raw['is_elite'] = (raw['overall'] >= 80).astype(int)
raw['main_position'] = raw['player_positions'].astype(str).str.split(',').str[0].str.strip()

print(f'Registros para DW: {len(raw)}')
print(f'Jugadores élite (overall >= 80): {raw["is_elite"].sum()} ({raw["is_elite"].mean()*100:.1f}%)')

Registros para DW: 18350
Jugadores élite (overall >= 80): 485 (2.6%)


In [15]:
con = duckdb.connect(DB_PATH)

con.execute("""
    CREATE OR REPLACE TABLE dim_jugador AS
    SELECT player_id, short_name, long_name, age, height_cm, weight_kg,
           nationality_name, preferred_foot, main_position
    FROM raw
""")
print(' dim_jugador creada')

con.execute("""
    CREATE OR REPLACE TABLE dim_club AS
    SELECT DISTINCT player_id, club_name, league_name, league_level,
           club_position, club_jersey_number
    FROM raw
""")
print(' dim_club creada')

con.execute("""
    CREATE OR REPLACE TABLE fact_rendimiento AS
    SELECT player_id, overall, potential, value_eur, wage_eur,
           pace, shooting, passing, dribbling, defending, physic,
           skill_moves, weak_foot, international_reputation, is_elite
    FROM raw
""")
print(' fact_rendimiento creada')

 dim_jugador creada
 dim_club creada
 fact_rendimiento creada


In [16]:
con.execute("""
    CREATE OR REPLACE VIEW olap_kpis AS
    SELECT COUNT(*) AS total_jugadores, SUM(is_elite) AS total_elite,
        ROUND(AVG(is_elite)*100, 2) AS pct_elite, ROUND(AVG(overall), 2) AS avg_overall,
        ROUND(AVG(potential), 2) AS avg_potential, ROUND(AVG(value_eur), 2) AS avg_value_eur,
        ROUND(AVG(wage_eur), 2) AS avg_wage_eur, ROUND(AVG(pace), 2) AS avg_pace,
        ROUND(AVG(shooting), 2) AS avg_shooting, ROUND(AVG(passing), 2) AS avg_passing,
        ROUND(AVG(dribbling), 2) AS avg_dribbling, ROUND(AVG(defending), 2) AS avg_defending,
        ROUND(AVG(physic), 2) AS avg_physic
    FROM fact_rendimiento
""")
print(' olap_kpis')

con.execute("""
    CREATE OR REPLACE VIEW olap_por_liga AS
    SELECT c.league_name, COUNT(*) AS total_jugadores, SUM(f.is_elite) AS total_elite,
        ROUND(AVG(f.is_elite)*100, 2) AS pct_elite, ROUND(AVG(f.overall), 2) AS avg_overall,
        ROUND(AVG(f.value_eur), 2) AS avg_value_eur, ROUND(AVG(f.wage_eur), 2) AS avg_wage_eur,
        ROUND(AVG(f.pace), 2) AS avg_pace, ROUND(AVG(f.shooting), 2) AS avg_shooting
    FROM fact_rendimiento f JOIN dim_club c USING (player_id)
    WHERE c.league_name IS NOT NULL
    GROUP BY c.league_name ORDER BY avg_overall DESC
""")
print(' olap_por_liga')

con.execute("""
    CREATE OR REPLACE VIEW olap_por_nacionalidad AS
    SELECT j.nationality_name, COUNT(*) AS total_jugadores, SUM(f.is_elite) AS total_elite,
        ROUND(AVG(f.is_elite)*100, 2) AS pct_elite, ROUND(AVG(f.overall), 2) AS avg_overall,
        ROUND(AVG(f.value_eur), 2) AS avg_value_eur
    FROM fact_rendimiento f JOIN dim_jugador j USING (player_id)
    GROUP BY j.nationality_name HAVING COUNT(*) >= 20
    ORDER BY avg_overall DESC
""")
print(' olap_por_nacionalidad')

con.execute("""
    CREATE OR REPLACE VIEW olap_por_posicion AS
    SELECT j.main_position, COUNT(*) AS total_jugadores,
        ROUND(AVG(f.overall), 2) AS avg_overall, ROUND(AVG(f.value_eur), 2) AS avg_value_eur,
        ROUND(AVG(f.pace), 2) AS avg_pace, ROUND(AVG(f.shooting), 2) AS avg_shooting,
        ROUND(AVG(f.passing), 2) AS avg_passing, ROUND(AVG(f.dribbling), 2) AS avg_dribbling,
        ROUND(AVG(f.defending), 2) AS avg_defending, ROUND(AVG(f.physic), 2) AS avg_physic
    FROM fact_rendimiento f JOIN dim_jugador j USING (player_id)
    GROUP BY j.main_position ORDER BY avg_overall DESC
""")
print(' olap_por_posicion')

 olap_kpis
 olap_por_liga
 olap_por_nacionalidad
 olap_por_posicion


In [17]:
print('=== KPIs Globales ===')
print(con.execute('SELECT * FROM olap_kpis').fetchdf().to_string())
print('\n=== Top 10 Ligas ===')
print(con.execute('SELECT * FROM olap_por_liga LIMIT 10').fetchdf().to_string())
print('\n=== Top 10 Nacionalidades ===')
print(con.execute('SELECT * FROM olap_por_nacionalidad LIMIT 10').fetchdf().to_string())
print('\n=== Por Posición ===')
print(con.execute('SELECT * FROM olap_por_posicion').fetchdf().to_string())

con.close()
print(f'\n DuckDB warehouse → {DB_PATH}')

=== KPIs Globales ===
   total_jugadores  total_elite  pct_elite  avg_overall  avg_potential  avg_value_eur  avg_wage_eur  avg_pace  avg_shooting  avg_passing  avg_dribbling  avg_defending  avg_physic
0            18350        485.0       2.64        65.82          71.09     1482729.84       5556.58     60.75         46.72        51.08          55.84          46.29       57.67

=== Top 10 Ligas ===
      league_name  total_jugadores  total_elite  pct_elite  avg_overall  avg_value_eur  avg_wage_eur  avg_pace  avg_shooting
0         La Liga              596        113.0      18.96        73.05     3163322.15      14003.52     60.90         52.54
1  Premier League              679        113.0      16.64        72.91     3141056.70      14694.26     62.55         51.24
2         Serie A              842         89.0      10.57        72.51     2844893.11      13686.64     62.44         52.44
3    Fortuna Liga               79          0.0       0.00        71.39     2737341.77        767.

---
## 4. Machine Learning

### Modelos:
1. **Regresión Lineal** → predecir `value_eur`
2. **Regresión Logística** → clasificar `is_elite` (overall ≥ 80)
3. **Random Forest** → clasificar `is_elite`

### Prevención de Data Leakage:
- `potential`, `wage_eur`, `overall` excluidos de features

In [18]:
FEATURE_COLS = [
    'age', 'height_cm', 'weight_kg',
    'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',
    'skill_moves', 'weak_foot', 'international_reputation',
    'preferred_foot', 'work_rate', 'body_type',
]

df['is_elite'] = (df_warehouse['overall'] >= 80).astype(int)

TARGET_VALUE = 'value_eur'
TARGET_ELITE = 'is_elite'

print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')
print(f'\nDistribución is_elite:')
print(df[TARGET_ELITE].value_counts())
print(f'Porcentaje élite: {df[TARGET_ELITE].mean()*100:.1f}%')

# Verificar limpieza
nan_check = df[FEATURE_COLS].isnull().sum()
nan_total = nan_check.sum()
print(f'\nNaN en features: {nan_total}')
if nan_total > 0:
    print(nan_check[nan_check > 0])
    raise ValueError('Hay NaN en features — revisar EDA')
print(' Features limpias')

Features (15): ['age', 'height_cm', 'weight_kg', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'skill_moves', 'weak_foot', 'international_reputation', 'preferred_foot', 'work_rate', 'body_type']

Distribución is_elite:
is_elite
0    17865
1      485
Name: count, dtype: int64
Porcentaje élite: 2.6%

NaN en features: 0
 Features limpias


In [19]:
X = df[FEATURE_COLS].copy()
y_value = df[TARGET_VALUE].copy()
y_elite = df[TARGET_ELITE].copy()

X_train, X_test, yv_train, yv_test, ye_train, ye_test = train_test_split(
    X, y_value, y_elite,
    test_size=0.2, random_state=42, stratify=y_elite
)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print(f'Élite en train: {ye_train.sum()} ({ye_train.mean()*100:.1f}%)')
print(f'Élite en test:  {ye_test.sum()} ({ye_test.mean()*100:.1f}%)')

Train: 14680 | Test: 3670
Élite en train: 388 (2.6%)
Élite en test:  97 (2.6%)


### 4a. Regresión Lineal — Predecir Valor de Mercado

In [20]:
lr = LinearRegression()
lr.fit(X_train, yv_train)
yv_pred = lr.predict(X_test)

rmse = np.sqrt(mean_squared_error(yv_test, yv_pred))
r2   = r2_score(yv_test, yv_pred)

print(f'[Linear Regression] → Predecir value_eur')
print(f'  RMSE = {rmse:.4f}')
print(f'  R²   = {r2:.4f}')

joblib.dump(lr, os.path.join(MODELS_DIR, 'linear_regression.joblib'))
print('  Modelo guardado ')

[Linear Regression] → Predecir value_eur
  RMSE = 0.8120
  R²   = 0.3433
  Modelo guardado 


### 4b. Regresión Logística — Clasificar Jugador Élite

In [21]:
log_clf = LogisticRegression(max_iter=1000, random_state=42)
log_clf.fit(X_train, ye_train)
ye_pred_log = log_clf.predict(X_test)

acc_log = accuracy_score(ye_test, ye_pred_log)
auc_log = roc_auc_score(ye_test, log_clf.predict_proba(X_test)[:, 1])

print(f'[Logistic Regression] → Clasificar is_elite')
print(f'  Accuracy = {acc_log:.4f}')
print(f'  AUC-ROC  = {auc_log:.4f}')
print()
print(classification_report(ye_test, ye_pred_log, target_names=['No Élite', 'Élite']))

joblib.dump(log_clf, os.path.join(MODELS_DIR, 'logistic_regression.joblib'))
print('Modelo guardado ')

[Logistic Regression] → Clasificar is_elite
  Accuracy = 0.9815
  AUC-ROC  = 0.9521

              precision    recall  f1-score   support

    No Élite       0.99      1.00      0.99      3573
       Élite       0.74      0.46      0.57        97

    accuracy                           0.98      3670
   macro avg       0.86      0.73      0.78      3670
weighted avg       0.98      0.98      0.98      3670

Modelo guardado 


### 4c. Random Forest — Clasificar Jugador Élite

In [22]:
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, ye_train)
ye_pred_rf = rf_clf.predict(X_test)

acc_rf = accuracy_score(ye_test, ye_pred_rf)
auc_rf = roc_auc_score(ye_test, rf_clf.predict_proba(X_test)[:, 1])

print(f'[Random Forest] → Clasificar is_elite')
print(f'  Accuracy = {acc_rf:.4f}')
print(f'  AUC-ROC  = {auc_rf:.4f}')
print()
print(classification_report(ye_test, ye_pred_rf, target_names=['No Élite', 'Élite']))

joblib.dump(rf_clf, os.path.join(MODELS_DIR, 'random_forest.joblib'))
print('Modelo guardado ')

[Random Forest] → Clasificar is_elite
  Accuracy = 0.9905
  AUC-ROC  = 0.9953

              precision    recall  f1-score   support

    No Élite       0.99      1.00      1.00      3573
       Élite       0.89      0.73      0.80        97

    accuracy                           0.99      3670
   macro avg       0.94      0.86      0.90      3670
weighted avg       0.99      0.99      0.99      3670

Modelo guardado 


In [23]:
meta = {
    'feature_cols': FEATURE_COLS,
    'target_value': TARGET_VALUE,
    'target_elite': TARGET_ELITE,
    'metrics': {
        'linear_regression': {'rmse': round(rmse, 4), 'r2': round(r2, 4)},
        'logistic_regression': {'accuracy': round(acc_log, 4), 'auc': round(auc_log, 4)},
        'random_forest': {'accuracy': round(acc_rf, 4), 'auc': round(auc_rf, 4)},
    }
}

with open(os.path.join(MODELS_DIR, 'model_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Metadatos guardados → {MODELS_DIR}/model_meta.json')
print(json.dumps(meta, indent=2))

Metadatos guardados → /Users/jaffetvicente/Downloads/Mineria de Datos/proyecto_corte_1/pipeline/models/model_meta.json
{
  "feature_cols": [
    "age",
    "height_cm",
    "weight_kg",
    "pace",
    "shooting",
    "passing",
    "dribbling",
    "defending",
    "physic",
    "skill_moves",
    "weak_foot",
    "international_reputation",
    "preferred_foot",
    "work_rate",
    "body_type"
  ],
  "target_value": "value_eur",
  "target_elite": "is_elite",
  "metrics": {
    "linear_regression": {
      "rmse": 0.812,
      "r2": 0.3433
    },
    "logistic_regression": {
      "accuracy": 0.9815,
      "auc": 0.9521
    },
    "random_forest": {
      "accuracy": 0.9905,
      "auc": 0.9953
    }
  }
}


---
## 5. Reporte Final

In [ ]:
print('=' * 60)
print('  PIPELINE COMPLETADO EXITOSAMENTE')
print('=' * 60)
print(f'  Dataset limpio   : {clean_path}')
print(f'  DuckDB warehouse : {DB_PATH}')
print(f'  Modelos ML       : {MODELS_DIR}/')
print(f'  Jugadores        : {len(df)}')
print(f'  Features         : {len(FEATURE_COLS)}')
print('─' * 60)
print('  MÉTRICAS:')
print(f'  Linear Regression  → RMSE={rmse:.4f}  R²={r2:.4f}')
print(f'  Logistic Regression→ Acc={acc_log:.4f}  AUC={auc_log:.4f}')
print(f'  Random Forest      → Acc={acc_rf:.4f}  AUC={auc_rf:.4f}')
print('─' * 60)
print('  Próximo paso: cd ../api && uvicorn main:app --reload')

  ✅ PIPELINE COMPLETADO EXITOSAMENTE
  Dataset limpio   : /Users/jaffetvicente/Downloads/Mineria de Datos/proyecto_corte_1/pipeline/data/players_clean.csv
  DuckDB warehouse : /Users/jaffetvicente/Downloads/Mineria de Datos/proyecto_corte_1/pipeline/warehouse.duckdb
  Modelos ML       : /Users/jaffetvicente/Downloads/Mineria de Datos/proyecto_corte_1/pipeline/models/
  Jugadores        : 18350
  Features         : 15
────────────────────────────────────────────────────────────
  MÉTRICAS:
  Linear Regression  → RMSE=0.8120  R²=0.3433
  Logistic Regression→ Acc=0.9815  AUC=0.9521
  Random Forest      → Acc=0.9905  AUC=0.9953
────────────────────────────────────────────────────────────
  Próximo paso: cd ../api && uvicorn main:app --reload
